# 03 — Cell QC, cell-cycle scoring, and generic 2-µm-bin `.obs` transfer

Run this notebook in the modern spatial/Scanpy environment after notebook 02.

It performs four tasks:

1. calculates standard cell-level count QC from the Proseg integer matrix;
2. scores mouse S and G2M cell-cycle programs on a temporary normalized copy;
3. maps source 2-µm bin centers into final Proseg polygons;
4. aggregates configured boolean, categorical, and continuous bin annotations
   into final cell-level `.obs` columns.

## Transfer semantics

### Boolean

For a source variable `is_positive`, output includes:

```text
is_positive_fraction_true
is_positive_n_valid_bins
```

### Categorical

For `pathology_region`, output includes:

```text
pathology_region_dominant
pathology_region_dominant_fraction
pathology_region_is_tie
pathology_region_n_valid_bins
pathology_region__fraction__<each category>
```

Exact ties for the highest bin count are labeled `mixed` by default.

### Continuous

The configuration selects one or more of:

```text
sum, mean, median
```

for each source variable.

The assignment rule is polygon containment of the 2-µm bin center. A boundary
fallback uses polygon intersection only for points lying exactly on a boundary.
The assignment and aggregation reports are saved for every sample.


## H5AD serialization fix

The previous run completed bin-to-polygon assignment and populated the new
`.obs` columns, but failed while writing:

```text
/uns/bin_annotation_transfer/variable_reports
```

because that field was a Python **list of dictionaries**. AnnData/HDF5 does not
serialize heterogeneous object lists reliably.

This revision keeps the full report in the external JSON file and stores an
H5AD-safe dictionary keyed by variable/type in `.uns`. It also writes a
per-sample transferred-`.obs` schema CSV and validates categorical fraction
sums before saving.

Existing bin-to-cell assignment `.npy` files remain reusable because the
assignment signature is unchanged.


In [1]:
# ---------------------------------------------------------------------
# Imports and configuration
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import hashlib
import json
import os
import sys
import re
import shutil
import traceback
import warnings
from pathlib import Path

import anndata as ad
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import spatialdata
from shapely import points
from shapely.strtree import STRtree

CONFIG_PATH = Path(
    os.environ.get(
        "VISIUMHD_PIPELINE_CONFIG",
        "/stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/"
        "derived_files/tbio8110_stardist_proseg_resolvi_v1/"
        "00_config/pipeline_config.json",
    )
)
CONFIG = json.loads(CONFIG_PATH.read_text())
CUSTOM_FUNCTION_DIR = Path(CONFIG["paths"]["functiondirs"])
if CUSTOM_FUNCTION_DIR.exists():
    if str(CUSTOM_FUNCTION_DIR) not in sys.path:
        sys.path.insert(0, str(CUSTOM_FUNCTION_DIR))
    print("Custom function directory enabled:", CUSTOM_FUNCTION_DIR)
else:
    warnings.warn(
        f"Configured functiondirs path does not exist: {CUSTOM_FUNCTION_DIR}. "
        "The built-in notebook helpers will be used."
    )
DERIVED_ROOT = Path(CONFIG["paths"]["derived_root"])
TEMP_ROOT = Path(CONFIG["paths"]["temp_root"])
CONFIG_ROOT = Path(CONFIG["paths"]["config_root"])
manifest = pd.read_csv(CONFIG_ROOT / "sample_manifest.csv")

PROSEG_ROOT = DERIVED_ROOT / "02_proseg"
TRANSFER_ROOT = DERIVED_ROOT / "03_cell_qc_obs_transfer"
TRANSFER_TEMP_ROOT = TEMP_ROOT / "03_cell_qc_obs_transfer"
TRANSFER_ROOT.mkdir(parents=True, exist_ok=True)
TRANSFER_TEMP_ROOT.mkdir(parents=True, exist_ok=True)

QC_TABLE_KEY = CONFIG["qc"]["table_key"]
QC_CLASS_COLUMN = CONFIG["qc"]["class_column"]
QC_CLASS_NORMALIZATION = CONFIG["qc"]["class_normalization"]
TRANSFER_SPEC = CONFIG["bin_obs_transfer"]

POINT_ASSIGNMENT_CHUNK_SIZE = 250_000
USE_BOUNDARY_INTERSECTS_FALLBACK = True
OVERWRITE_ASSIGNMENT = False
OVERWRITE_OUTPUT = False
CONTINUE_ON_ERROR = True

# Inclusive ResolVI eligibility. These are technical requirements, not a harsh
# biological quality filter.
MIN_COUNTS_FOR_RESOLVI = 1
MIN_GENES_FOR_RESOLVI = 1
APPLY_MAX_MT_FILTER = False
MAX_PCT_MT = 25.0

CELL_CYCLE_TARGET_SUM = 10_000.0
H5AD_COMPRESSION = "lzf"
PIPELINE_VERSION = "reusable-cell-qc-binobs-transfer-v1"
OUTPUT_SCHEMA_VERSION = "h5ad-safe-uns-v2"

print("Transfer specification:")
print(json.dumps(TRANSFER_SPEC, indent=2))
print("Samples:", manifest["sample"].astype(str).tolist())


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_

Custom function directory enabled: /host_root/nethome/reny28/Projects/Custom_functions/python_functions
Transfer specification:
{
  "bin_scope": "all_bins",
  "boolean": {},
  "categorical": {
    "qc_class": {
      "tie_label": "mixed",
      "categories": [
        "tissue-high_transcript-high",
        "tissue-high_transcript-low",
        "tissue-low_transcript-high",
        "tissue-low_transcript-low"
      ],
      "normalize_qc_class": true,
      "max_categories": 20
    }
  },
  "continuous": {}
}
Samples: ['Ada-1', 'Ada-3R', 'Ada-4R', 'Ada-6', 'Ada-7', 'Ada-8', 'Ada-9', 'Ada-11R', 'Ada-12', 'Ada-14R', 'Ada-15', 'Ada-16']


In [2]:
# ---------------------------------------------------------------------
# Paths and general utilities
# ---------------------------------------------------------------------
def paths_for_sample(row: pd.Series) -> dict[str, Path]:
    sample = str(row["sample"])
    proseg = PROSEG_ROOT / sample
    temp = TRANSFER_TEMP_ROOT / sample
    durable = TRANSFER_ROOT / sample
    temp.mkdir(parents=True, exist_ok=True)
    durable.mkdir(parents=True, exist_ok=True)
    return {
        "sample": sample,
        "zarr": Path(row["zarr_path"]),
        "aligned_qc": Path(row["aligned_qc_parquet"]),
        "proseg_cdata": proseg / f"{sample}_proseg_qcclass_cdata.h5ad",
        "boundaries": (
            proseg / f"{sample}_proseg_qcclass_cell_boundaries.parquet"
        ),
        "assignment": temp / f"{sample}_bin_to_proseg_cell_index.npy",
        "assignment_summary": (
            durable / f"{sample}_bin_to_cell_assignment_summary.json"
        ),
        "transfer_report": (
            durable / f"{sample}_bin_obs_transfer_report.json"
        ),
        "obs_transfer_schema": (
            durable / f"{sample}_transferred_obs_schema.csv"
        ),
        "annotated": durable / f"{sample}_proseg_qc_annotated.h5ad",
        "resolvi_input": durable / f"{sample}_proseg_resolvi_input.h5ad",
        "qc_summary": durable / f"{sample}_cell_qc_summary.csv",
        "cell_cycle_summary": (
            durable / f"{sample}_cell_cycle_summary.csv"
        ),
        "success": durable / f"{sample}_qc_transfer_SUCCESS.json",
        "failure": durable / f"{sample}_qc_transfer_FAILURE.json",
    }


def atomic_json(payload, path: Path):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str))
    temp.replace(path)


def sanitize_none(value):
    """
    Backward-compatible scalar/container cleanup for JSON-style payloads.

    Lists of dictionaries are intentionally handled by
    ``reports_to_uns_mapping`` rather than left as heterogeneous object arrays.
    """
    if value is None:
        return ""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {
            str(key): sanitize_none(item)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [
            sanitize_none(item)
            for item in value
        ]
    if isinstance(value, tuple):
        return tuple(
            sanitize_none(item)
            for item in value
        )
    return value


def reports_to_uns_mapping(reports):
    """
    Convert a list of heterogeneous report dictionaries into an H5AD-safe
    nested mapping.

    AnnData can serialize nested dictionaries, but a list of dictionaries is
    interpreted as a heterogeneous object array and can fail with:

        TypeError: Can't implicitly convert non-string objects to strings
    """
    output = {}

    for index, report in enumerate(reports):
        report = sanitize_none(dict(report))
        report_type = slugify(
            report.get("type", "report")
        )
        variable = slugify(
            report.get("variable", f"item_{index}")
        )

        key = f"{index:03d}_{report_type}_{variable}"

        # Defend against accidental duplicate keys.
        if key in output:
            key = f"{key}_{index}"

        output[key] = report

    return output


def sanitize_uns_for_h5ad(value):
    """
    Recursively convert remaining H5AD-incompatible containers.

    This preserves homogeneous scalar lists, while converting any list that
    contains mappings into a deterministic mapping.
    """
    if value is None:
        return ""

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): sanitize_uns_for_h5ad(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        cleaned = [
            sanitize_uns_for_h5ad(item)
            for item in value
        ]

        if any(
            isinstance(item, dict)
            for item in cleaned
        ):
            return {
                f"item_{index:04d}": item
                for index, item in enumerate(cleaned)
            }

        return cleaned

    return value


def atomic_write_h5ad(
    adata_obj: ad.AnnData,
    path: Path,
):
    """
    Atomically write and read back an H5AD after sanitizing `.uns`.
    """
    temp = path.with_name(
        path.stem + ".tmp.h5ad"
    )
    temp.unlink(missing_ok=True)

    adata_obj.uns = sanitize_uns_for_h5ad(
        dict(adata_obj.uns)
    )

    adata_obj.write_h5ad(
        temp,
        compression=H5AD_COMPRESSION,
    )

    check = ad.read_h5ad(
        temp,
        backed="r",
    )

    try:
        if check.shape != adata_obj.shape:
            raise ValueError(
                "H5AD read-back shape mismatch."
            )

        # Confirm that the new annotation block itself is readable.
        if "bin_annotation_transfer" not in check.uns:
            raise KeyError(
                "H5AD read-back lacks "
                "uns['bin_annotation_transfer']."
            )
    finally:
        check.file.close()

    temp.replace(path)


def clear_local_crs(gdf: gpd.GeoDataFrame):
    gdf = gdf.copy()
    if gdf.crs is not None:
        try:
            gdf = gdf.set_crs(None, allow_override=True)
        except Exception:
            gdf.crs = None
    return gdf


def normalize_qc_class(values: pd.Series):
    result = values.astype("string").str.strip()
    for observed, canonical in QC_CLASS_NORMALIZATION.items():
        result = result.str.replace(observed, canonical, regex=False)
    return result


def slugify(value: str) -> str:
    slug = re.sub(r"[^0-9A-Za-z]+", "_", str(value)).strip("_").lower()
    return slug or "empty"


In [3]:
# ---------------------------------------------------------------------
# Standard QC and mouse cell-cycle scoring
# ---------------------------------------------------------------------
MOUSE_S_GENES = [
    "Mcm5", "Pcna", "Tyms", "Fen1", "Mcm2", "Mcm4", "Rrm1", "Ung",
    "Gins2", "Mcm6", "Cdca7", "Dtl", "Prim1", "Uhrf1", "Mlf1ip",
    "Hells", "Rfc2", "Rpa2", "Nasp", "Rad51ap1", "Gmnn", "Wdr76",
    "Slbp", "Ccne2", "Ubr7", "Pold3", "Msh2", "Atad2", "Rad51",
    "Rrm2", "Cdc45", "Cdc6", "Exo1", "Tipin", "Dscc1", "Blm",
    "Casp8ap2", "Usp1", "Clspn", "Pola1", "Chaf1b", "Brip1", "E2f8",
]

MOUSE_G2M_GENES = [
    "Hmgb2", "Cdk1", "Nusap1", "Ube2c", "Birc5", "Tpx2", "Top2a",
    "Ndc80", "Cks2", "Nuf2", "Cks1b", "Mki67", "Tmpo", "Cenpf",
    "Tacc3", "Fam64a", "Smc4", "Ccnb2", "Ckap2l", "Ckap2", "Aurkb",
    "Bub1", "Kif11", "Anp32e", "Tubb4b", "Gtse1", "Kif20b", "Hjurp",
    "Cdca3", "Hn1", "Cdc20", "Ttk", "Cdc25c", "Kif2c", "Rangap1",
    "Ncapd2", "Dlgap5", "Cdca2", "Cdca8", "Ect2", "Kif23", "Hmmr",
    "Aurka", "Psrc1", "Anln", "Lbr", "Ckap5", "Cenpe", "Ctcf", "Nek2",
    "G2e3", "Gas2l3", "Cbx5", "Cenpa",
]


def add_standard_qc(adata_obj: ad.AnnData):
    if "gene" in adata_obj.var.columns:
        symbols = pd.Index(adata_obj.var["gene"].astype(str))
    else:
        symbols = pd.Index(adata_obj.var_names.astype(str))
    upper = symbols.str.upper()

    adata_obj.var["mt"] = np.asarray(
        upper.str.startswith(("MT-", "MT_")), dtype=bool
    )
    adata_obj.var["ribo"] = np.asarray(
        upper.str.startswith(("RPS", "RPL")), dtype=bool
    )
    adata_obj.var["hb"] = np.asarray(
        upper.str.match(r"^HB(?!P)"), dtype=bool
    )
    sc.pp.calculate_qc_metrics(
        adata_obj,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )
    adata_obj.obs["n_counts"] = adata_obj.obs["total_counts"].to_numpy()
    adata_obj.obs["n_genes"] = adata_obj.obs[
        "n_genes_by_counts"
    ].to_numpy()


def resolve_gene_symbols(adata_obj: ad.AnnData):
    if "gene" in adata_obj.var.columns:
        symbols = adata_obj.var["gene"].astype(str).to_numpy()
    else:
        symbols = adata_obj.var_names.astype(str).to_numpy()
    lookup = {}
    for var_name, symbol in zip(adata_obj.var_names, symbols, strict=True):
        lookup.setdefault(str(symbol).upper(), str(var_name))
    return lookup


def add_cell_cycle_scores(adata_obj: ad.AnnData):
    adata_obj.obs["S_score"] = np.nan
    adata_obj.obs["G2M_score"] = np.nan
    # Keep phase as an object array during assignment. Converting to
    # Categorical before writing S/G2M/G1 values would reject unseen categories.
    adata_obj.obs["phase"] = np.full(
        adata_obj.n_obs,
        "unscored",
        dtype=object,
    )
    adata_obj.obs["cell_cycle_scored"] = False

    eligible = adata_obj.obs["total_counts"].to_numpy() > 0
    if not eligible.any():
        warnings.warn("No nonzero cells are available for cell-cycle scoring.")
        return {"n_s_genes": 0, "n_g2m_genes": 0, "n_cells_scored": 0}

    lookup = resolve_gene_symbols(adata_obj)
    s_genes = [lookup[g.upper()] for g in MOUSE_S_GENES if g.upper() in lookup]
    g2m_genes = [
        lookup[g.upper()] for g in MOUSE_G2M_GENES if g.upper() in lookup
    ]

    report = {
        "n_s_genes": int(len(s_genes)),
        "n_g2m_genes": int(len(g2m_genes)),
        "s_genes": s_genes,
        "g2m_genes": g2m_genes,
        "n_cells_scored": int(eligible.sum()),
    }
    if len(s_genes) < 5 or len(g2m_genes) < 5:
        warnings.warn(
            "Too few cell-cycle genes are present for reliable scoring: "
            f"S={len(s_genes)}, G2M={len(g2m_genes)}"
        )
        return report

    temporary = adata_obj[eligible].copy()
    sc.pp.normalize_total(temporary, target_sum=CELL_CYCLE_TARGET_SUM)
    sc.pp.log1p(temporary)
    sc.tl.score_genes_cell_cycle(
        temporary,
        s_genes=s_genes,
        g2m_genes=g2m_genes,
        use_raw=False,
    )

    target_names = adata_obj.obs_names[eligible]
    adata_obj.obs.loc[target_names, "S_score"] = temporary.obs[
        "S_score"
    ].to_numpy()
    adata_obj.obs.loc[target_names, "G2M_score"] = temporary.obs[
        "G2M_score"
    ].to_numpy()
    adata_obj.obs.loc[target_names, "phase"] = temporary.obs[
        "phase"
    ].astype(str).to_numpy()
    adata_obj.obs.loc[target_names, "cell_cycle_scored"] = True
    adata_obj.obs["phase"] = pd.Categorical(adata_obj.obs["phase"])

    del temporary
    gc.collect()
    return report


In [4]:
# ---------------------------------------------------------------------
# Chunked bin-center to polygon assignment
# ---------------------------------------------------------------------
def load_and_align_boundaries(cdata: ad.AnnData, path: Path):
    gdf = clear_local_crs(gpd.read_parquet(path))
    if "cell" in gdf.columns:
        gdf["cell"] = pd.to_numeric(gdf["cell"], errors="raise").astype(np.int64)
        gdf.index = gdf["cell"].astype(str)
    gdf.index = gdf.index.astype(str)

    missing = cdata.obs_names.difference(gdf.index)
    if len(missing):
        raise ValueError(
            f"{len(missing)} H5AD cells lack a polygon; examples={missing[:5].tolist()}"
        )
    gdf = gdf.reindex(cdata.obs_names)
    if gdf.geometry.isna().any():
        raise ValueError("Boundary reindexing introduced missing geometries.")
    return gdf


def choose_smallest_polygon_per_point(local_index, polygon_index, areas):
    if len(local_index) == 0:
        return local_index, polygon_index, 0
    order = np.lexsort((areas[polygon_index], local_index))
    local_sorted = local_index[order]
    polygon_sorted = polygon_index[order]
    first = np.r_[True, local_sorted[1:] != local_sorted[:-1]]
    multiple = int(len(local_sorted) - first.sum())
    return local_sorted[first], polygon_sorted[first], multiple


def build_bin_assignment(
    x_um,
    y_um,
    polygons,
    output_path: Path,
    summary_path: Path,
    signature: dict,
):
    signature_hash = hashlib.sha256(
        json.dumps(signature, sort_keys=True).encode()
    ).hexdigest()

    if output_path.exists() and summary_path.exists() and not OVERWRITE_ASSIGNMENT:
        existing = json.loads(summary_path.read_text())
        if existing.get("signature_hash") == signature_hash:
            mapping = np.load(output_path, mmap_mode="r")
            if len(mapping) == len(x_um):
                print("Reusing bin-to-cell assignment:", output_path)
                return mapping, existing
        raise RuntimeError(
            "Existing bin assignment does not match current inputs. Set "
            "OVERWRITE_ASSIGNMENT=True to rebuild it."
        )

    output_path.unlink(missing_ok=True)
    assignment = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=np.int32,
        shape=(len(x_um),),
    )
    assignment[:] = -1

    geometry_array = np.asarray(polygons.geometry.values, dtype=object)
    areas = polygons.geometry.area.to_numpy(dtype=np.float64)
    tree = STRtree(geometry_array)

    multiple_within = 0
    multiple_boundary = 0
    assigned_within = 0
    assigned_boundary = 0

    for start in range(0, len(x_um), POINT_ASSIGNMENT_CHUNK_SIZE):
        stop = min(start + POINT_ASSIGNMENT_CHUNK_SIZE, len(x_um))
        chunk_points = points(x_um[start:stop], y_um[start:stop])

        pairs = tree.query(chunk_points, predicate="within")
        if pairs.size:
            local, polygon, duplicates = (
                choose_smallest_polygon_per_point(
                    pairs[0].astype(np.int64),
                    pairs[1].astype(np.int64),
                    areas,
                )
            )
            assignment[start + local] = polygon.astype(np.int32)
            assigned_within += len(local)
            multiple_within += duplicates

        if USE_BOUNDARY_INTERSECTS_FALLBACK:
            current = np.asarray(assignment[start:stop])
            unresolved = np.flatnonzero(current < 0)
            if len(unresolved):
                boundary_pairs = tree.query(
                    chunk_points[unresolved],
                    predicate="intersects",
                )
                if boundary_pairs.size:
                    local_unresolved, polygon, duplicates = (
                        choose_smallest_polygon_per_point(
                            boundary_pairs[0].astype(np.int64),
                            boundary_pairs[1].astype(np.int64),
                            areas,
                        )
                    )
                    original_local = unresolved[local_unresolved]
                    assignment[start + original_local] = polygon.astype(np.int32)
                    assigned_boundary += len(original_local)
                    multiple_boundary += duplicates

        if start % (POINT_ASSIGNMENT_CHUNK_SIZE * 10) == 0:
            print(f"  assigned bins {start:,}–{stop:,} / {len(x_um):,}")

    assignment.flush()
    mapped = np.asarray(assignment) >= 0
    summary = {
        **signature,
        "signature_hash": signature_hash,
        "n_bins_considered": int(len(x_um)),
        "n_cells": int(len(polygons)),
        "n_bins_assigned": int(mapped.sum()),
        "fraction_bins_assigned": float(mapped.mean()),
        "n_assigned_by_within": int(assigned_within),
        "n_assigned_by_boundary_fallback": int(assigned_boundary),
        "n_extra_multiple_within_matches": int(multiple_within),
        "n_extra_multiple_boundary_matches": int(multiple_boundary),
        "predicate": "within_then_intersects_boundary_fallback",
    }
    atomic_json(summary, summary_path)
    return assignment, summary


In [5]:
# ---------------------------------------------------------------------
# Generic boolean, categorical, and continuous aggregators
# ---------------------------------------------------------------------
def add_boolean_transfer(
    cdata,
    variable,
    values,
    assignment,
    spec,
):
    true_values = spec.get(
        "true_values", [True, 1, "true", "True", "TRUE"]
    )
    false_values = spec.get(
        "false_values", [False, 0, "false", "False", "FALSE"]
    )
    true_tokens = {str(v).strip().lower() for v in true_values}
    false_tokens = {str(v).strip().lower() for v in false_values}

    series = pd.Series(values)
    nonmissing = series.notna().to_numpy()
    tokens = series.astype("string").str.strip().str.lower()
    is_true = tokens.isin(true_tokens).to_numpy()
    is_false = tokens.isin(false_tokens).to_numpy()
    unknown = nonmissing & ~(is_true | is_false)
    if unknown.any():
        examples = series[unknown].astype(str).unique()[:10].tolist()
        raise ValueError(
            f"Boolean variable {variable!r} contains unconfigured values: {examples}"
        )

    valid = (assignment >= 0) & nonmissing
    cell_index = np.asarray(assignment[valid], dtype=np.int64)
    valid_count = np.bincount(cell_index, minlength=cdata.n_obs)
    true_count = np.bincount(
        np.asarray(assignment[valid & is_true], dtype=np.int64),
        minlength=cdata.n_obs,
    )
    fraction = np.full(cdata.n_obs, np.nan, dtype=np.float32)
    np.divide(true_count, valid_count, out=fraction, where=valid_count > 0)

    cdata.obs[f"{variable}_fraction_true"] = fraction
    cdata.obs[f"{variable}_n_valid_bins"] = valid_count
    return {
        "variable": variable,
        "type": "boolean",
        "n_source_nonmissing": int(nonmissing.sum()),
        "n_cells_with_valid_bins": int((valid_count > 0).sum()),
    }


def category_slug_map(categories):
    mapping = {}
    used = set()
    for category in categories:
        base = slugify(category)
        slug = base
        if slug in used:
            suffix = hashlib.sha1(str(category).encode()).hexdigest()[:8]
            slug = f"{base}_{suffix}"
        mapping[str(category)] = slug
        used.add(slug)
    return mapping


def add_categorical_transfer(
    cdata,
    variable,
    values,
    assignment,
    spec,
):
    series = pd.Series(values, dtype="string")
    if spec.get("normalize_qc_class", False):
        series = normalize_qc_class(series)

    nonmissing = series.notna().to_numpy()
    observed = sorted(series.dropna().astype(str).unique().tolist())
    categories = spec.get("categories")
    categories = observed if categories is None else [str(v) for v in categories]
    unexpected = sorted(set(observed) - set(categories))
    if unexpected:
        raise ValueError(
            f"Categorical variable {variable!r} has categories absent from the "
            f"configured universe: {unexpected}"
        )
    if len(categories) > int(spec.get("max_categories", 100)):
        raise ValueError(
            f"{variable!r} has {len(categories)} categories, exceeding max_categories."
        )

    valid_assignment = np.asarray(assignment)
    valid = (valid_assignment >= 0) & nonmissing
    valid_cells = valid_assignment[valid].astype(np.int64, copy=False)
    valid_count = np.bincount(valid_cells, minlength=cdata.n_obs)

    top_count = np.zeros(cdata.n_obs, dtype=np.int64)
    top_index = np.full(cdata.n_obs, -1, dtype=np.int32)
    tie_count = np.zeros(cdata.n_obs, dtype=np.int16)
    slugs = category_slug_map(categories)

    for category_index, category in enumerate(categories):
        mask = valid & (series.astype(str).to_numpy() == category)
        counts = np.bincount(
            valid_assignment[mask].astype(np.int64, copy=False),
            minlength=cdata.n_obs,
        )
        fraction = np.full(cdata.n_obs, np.nan, dtype=np.float32)
        np.divide(counts, valid_count, out=fraction, where=valid_count > 0)
        cdata.obs[
            f"{variable}__fraction__{slugs[category]}"
        ] = fraction

        greater = counts > top_count
        equal_positive = (counts == top_count) & (counts > 0) & ~greater
        top_count[greater] = counts[greater]
        top_index[greater] = category_index
        tie_count[greater] = 1
        tie_count[equal_positive] += 1

    dominant = np.empty(cdata.n_obs, dtype=object)
    dominant[:] = None
    has_value = valid_count > 0
    unique_top = has_value & (tie_count == 1)
    tied = has_value & (tie_count > 1)
    category_array = np.asarray(categories, dtype=object)
    dominant[unique_top] = category_array[top_index[unique_top]]
    dominant[tied] = str(spec.get("tie_label", "mixed"))

    dominant_fraction = np.full(cdata.n_obs, np.nan, dtype=np.float32)
    np.divide(
        top_count,
        valid_count,
        out=dominant_fraction,
        where=valid_count > 0,
    )

    cdata.obs[f"{variable}_dominant"] = pd.Categorical(dominant)
    cdata.obs[f"{variable}_dominant_fraction"] = dominant_fraction
    cdata.obs[f"{variable}_is_tie"] = tied
    cdata.obs[f"{variable}_n_valid_bins"] = valid_count

    return {
        "variable": variable,
        "type": "categorical",
        "categories": categories,
        "category_column_slugs": slugs,
        "n_source_nonmissing": int(nonmissing.sum()),
        "n_cells_with_valid_bins": int(has_value.sum()),
        "n_cells_tied": int(tied.sum()),
    }


def add_continuous_transfer(
    cdata,
    variable,
    values,
    assignment,
    modes,
):
    modes = [modes] if isinstance(modes, str) else list(modes)
    source = pd.Series(values)
    numeric = pd.to_numeric(source, errors="coerce")
    invalid = source.notna() & numeric.isna()
    if invalid.any():
        examples = source[invalid].astype(str).unique()[:10].tolist()
        raise ValueError(
            f"Continuous variable {variable!r} has nonnumeric values: {examples}"
        )

    assignment_array = np.asarray(assignment)
    valid = (assignment_array >= 0) & numeric.notna().to_numpy()
    cells = assignment_array[valid].astype(np.int64, copy=False)
    vals = numeric.to_numpy(dtype=np.float64)[valid]
    valid_count = np.bincount(cells, minlength=cdata.n_obs)
    cdata.obs[f"{variable}_n_valid_bins"] = valid_count

    if "sum" in modes or "mean" in modes:
        sums = np.bincount(cells, weights=vals, minlength=cdata.n_obs)
        if "sum" in modes:
            output = sums.astype(np.float64)
            output[valid_count == 0] = np.nan
            cdata.obs[f"{variable}_sum"] = output
        if "mean" in modes:
            output = np.full(cdata.n_obs, np.nan, dtype=np.float64)
            np.divide(sums, valid_count, out=output, where=valid_count > 0)
            cdata.obs[f"{variable}_mean"] = output

    if "median" in modes:
        frame = pd.DataFrame({"cell": cells, "value": vals})
        medians = frame.groupby("cell", sort=False)["value"].median()
        output = np.full(cdata.n_obs, np.nan, dtype=np.float64)
        output[medians.index.to_numpy(dtype=np.int64)] = medians.to_numpy()
        cdata.obs[f"{variable}_median"] = output
        del frame

    return {
        "variable": variable,
        "type": "continuous",
        "modes": modes,
        "n_source_nonmissing": int(numeric.notna().sum()),
        "n_cells_with_valid_bins": int((valid_count > 0).sum()),
    }



def expected_transfer_columns(transfer_spec):
    """Return the configured cell-level `.obs` columns."""
    columns = []

    for variable in transfer_spec["boolean"]:
        columns.extend(
            [
                f"{variable}_fraction_true",
                f"{variable}_n_valid_bins",
            ]
        )

    for variable, spec in transfer_spec["categorical"].items():
        categories = spec.get("categories")

        columns.extend(
            [
                f"{variable}_dominant",
                f"{variable}_dominant_fraction",
                f"{variable}_is_tie",
                f"{variable}_n_valid_bins",
            ]
        )

        if categories is not None:
            slugs = category_slug_map(
                [str(value) for value in categories]
            )
            columns.extend(
                [
                    f"{variable}__fraction__{slugs[str(category)]}"
                    for category in categories
                ]
            )

    for variable, modes in transfer_spec["continuous"].items():
        modes = (
            [modes]
            if isinstance(modes, str)
            else list(modes)
        )

        columns.append(
            f"{variable}_n_valid_bins"
        )

        for mode in modes:
            columns.append(
                f"{variable}_{mode}"
            )

    return list(dict.fromkeys(columns))


def validate_transferred_obs(
    cdata,
    transfer_spec,
):
    """
    Validate the populated `.obs` transfer columns before H5AD writing.

    In particular, categorical per-category fractions should sum to one for
    cells that received one or more nonmissing source bins.
    """
    expected = expected_transfer_columns(
        transfer_spec
    )

    missing = [
        column
        for column in expected
        if column not in cdata.obs.columns
    ]

    if missing:
        raise KeyError(
            "Configured transfer columns are missing from cdata.obs: "
            f"{missing}"
        )

    report = {
        "n_expected_columns": int(len(expected)),
        "expected_columns": expected,
        "categorical_checks": {},
    }

    for variable, spec in transfer_spec[
        "categorical"
    ].items():
        categories = spec.get("categories")

        if categories is None:
            # Infer the already-created category fraction columns.
            prefix = f"{variable}__fraction__"
            fraction_columns = [
                column
                for column in cdata.obs.columns
                if column.startswith(prefix)
            ]
        else:
            slugs = category_slug_map(
                [str(value) for value in categories]
            )
            fraction_columns = [
                f"{variable}__fraction__{slugs[str(category)]}"
                for category in categories
            ]

        valid_count_column = (
            f"{variable}_n_valid_bins"
        )
        dominant_fraction_column = (
            f"{variable}_dominant_fraction"
        )

        valid_cells = (
            cdata.obs[valid_count_column]
            .to_numpy()
            > 0
        )

        if fraction_columns:
            fractions = (
                cdata.obs[fraction_columns]
                .to_numpy(dtype=np.float64)
            )

            row_sum = np.nansum(
                fractions,
                axis=1,
            )
            maximum = np.nanmax(
                fractions,
                axis=1,
            )

            if valid_cells.any():
                maximum_sum_error = float(
                    np.max(
                        np.abs(
                            row_sum[valid_cells]
                            - 1.0
                        )
                    )
                )

                maximum_dominant_error = float(
                    np.max(
                        np.abs(
                            maximum[valid_cells]
                            - cdata.obs[
                                dominant_fraction_column
                            ]
                            .to_numpy(dtype=np.float64)[
                                valid_cells
                            ]
                        )
                    )
                )
            else:
                maximum_sum_error = 0.0
                maximum_dominant_error = 0.0

            if maximum_sum_error > 1e-5:
                raise ValueError(
                    f"{variable}: categorical fractions do not sum to one; "
                    f"maximum error={maximum_sum_error}"
                )

            if maximum_dominant_error > 1e-5:
                raise ValueError(
                    f"{variable}: dominant fraction does not match the "
                    f"maximum category fraction; maximum error="
                    f"{maximum_dominant_error}"
                )

            report["categorical_checks"][
                variable
            ] = {
                "fraction_columns": fraction_columns,
                "n_cells_with_valid_bins": int(
                    valid_cells.sum()
                ),
                "maximum_fraction_sum_error": (
                    maximum_sum_error
                ),
                "maximum_dominant_fraction_error": (
                    maximum_dominant_error
                ),
            }

    return report


def build_obs_transfer_schema(
    cdata,
    transfer_spec,
):
    """
    Summarize the transferred `.obs` fields for audit and quick inspection.
    """
    columns = expected_transfer_columns(
        transfer_spec
    )

    rows = []

    for column in columns:
        series = cdata.obs[column]

        row = {
            "column": column,
            "dtype": str(series.dtype),
            "n_cells": int(len(series)),
            "n_missing": int(series.isna().sum()),
            "fraction_missing": float(
                series.isna().mean()
            ),
            "n_unique_nonmissing": int(
                series.nunique(dropna=True)
            ),
            "minimum": np.nan,
            "maximum": np.nan,
        }

        if pd.api.types.is_numeric_dtype(
            series.dtype
        ):
            numeric = pd.to_numeric(
                series,
                errors="coerce",
            )
            if numeric.notna().any():
                row["minimum"] = float(
                    numeric.min()
                )
                row["maximum"] = float(
                    numeric.max()
                )

        rows.append(row)

    return pd.DataFrame(rows)


In [6]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(row: pd.Series):
    sample = str(row["sample"])
    p = paths_for_sample(row)

    if (
        p["success"].exists()
        and p["annotated"].exists()
        and p["resolvi_input"].exists()
        and not OVERWRITE_OUTPUT
    ):
        print("Reusing completed QC/transfer output:", sample)
        return json.loads(p["success"].read_text())

    for required in [
        p["zarr"],
        p["aligned_qc"],
        p["proseg_cdata"],
        p["boundaries"],
    ]:
        if not required.exists():
            raise FileNotFoundError(required)

    cdata = ad.read_h5ad(p["proseg_cdata"])
    cdata.obs_names = cdata.obs_names.astype(str)
    cdata.obs["sample"] = pd.Categorical(
        np.repeat(sample, cdata.n_obs)
    )
    gdf = load_and_align_boundaries(cdata, p["boundaries"])

    aligned = pd.read_parquet(p["aligned_qc"])
    if TRANSFER_SPEC["bin_scope"] == "qc_keep_only":
        aligned = aligned.loc[aligned["qc_keep"].astype(bool)].copy()
    elif TRANSFER_SPEC["bin_scope"] != "all_bins":
        raise ValueError(TRANSFER_SPEC["bin_scope"])
    aligned.reset_index(drop=True, inplace=True)

    signature = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        "bin_scope": TRANSFER_SPEC["bin_scope"],
        "n_bins": int(len(aligned)),
        "n_cells": int(cdata.n_obs),
        "boundary_path": str(p["boundaries"]),
        "boundary_size": int(p["boundaries"].stat().st_size),
        "aligned_qc_path": str(p["aligned_qc"]),
        "aligned_qc_size": int(p["aligned_qc"].stat().st_size),
    }
    assignment, assignment_report = build_bin_assignment(
        aligned["proseg_x_um"].to_numpy(dtype=np.float64),
        aligned["proseg_y_um"].to_numpy(dtype=np.float64),
        gdf,
        p["assignment"],
        p["assignment_summary"],
        signature,
    )

    mapped = np.asarray(assignment) >= 0
    cdata.obs["n_annotation_bins"] = np.bincount(
        np.asarray(assignment[mapped], dtype=np.int64),
        minlength=cdata.n_obs,
    )

    # Read only the configured source columns from the QC SpatialData table.
    requested_variables = list(TRANSFER_SPEC["boolean"])
    requested_variables += list(TRANSFER_SPEC["categorical"])
    requested_variables += list(TRANSFER_SPEC["continuous"])
    requested_variables = list(dict.fromkeys(requested_variables))

    sdata = spatialdata.read_zarr(p["zarr"])
    if QC_TABLE_KEY not in sdata.tables:
        raise KeyError(QC_TABLE_KEY)
    table = sdata.tables[QC_TABLE_KEY]
    table.obs_names = table.obs_names.astype(str)

    missing_variables = [
        variable
        for variable in requested_variables
        if variable not in table.obs.columns
    ]
    if missing_variables:
        raise KeyError(
            f"Configured transfer variables absent from {sample}: {missing_variables}"
        )

    source_obs = table.obs[requested_variables].copy()
    source_obs.index = table.obs_names
    source_obs = source_obs.reindex(aligned["barcode"].astype(str))

    transfer_reports = []
    for variable, spec in TRANSFER_SPEC["boolean"].items():
        transfer_reports.append(
            add_boolean_transfer(
                cdata,
                variable,
                source_obs[variable].to_numpy(),
                assignment,
                spec,
            )
        )
    for variable, spec in TRANSFER_SPEC["categorical"].items():
        transfer_reports.append(
            add_categorical_transfer(
                cdata,
                variable,
                source_obs[variable].to_numpy(),
                assignment,
                spec,
            )
        )
    for variable, modes in TRANSFER_SPEC["continuous"].items():
        transfer_reports.append(
            add_continuous_transfer(
                cdata,
                variable,
                source_obs[variable].to_numpy(),
                assignment,
                modes,
            )
        )

    obs_transfer_validation = (
        validate_transferred_obs(
            cdata,
            TRANSFER_SPEC,
        )
    )

    obs_transfer_schema = (
        build_obs_transfer_schema(
            cdata,
            TRANSFER_SPEC,
        )
    )
    obs_transfer_schema.to_csv(
        p["obs_transfer_schema"],
        index=False,
    )

    print("Transferred .obs schema:")
    display(obs_transfer_schema)

    add_standard_qc(cdata)
    cell_cycle_report = add_cell_cycle_scores(cdata)

    finite_spatial = np.isfinite(
        np.asarray(cdata.obsm["spatial"])
    ).all(axis=1)
    cdata.obs["qc_warning_high_mito"] = (
        cdata.obs["pct_counts_mt"].to_numpy() > MAX_PCT_MT
    )
    cdata.obs["resolvi_eligible"] = (
        finite_spatial
        & (
            cdata.obs["total_counts"].to_numpy()
            >= MIN_COUNTS_FOR_RESOLVI
        )
        & (
            cdata.obs["n_genes_by_counts"].to_numpy()
            >= MIN_GENES_FOR_RESOLVI
        )
    )
    if APPLY_MAX_MT_FILTER:
        cdata.obs["resolvi_eligible"] &= ~cdata.obs[
            "qc_warning_high_mito"
        ].to_numpy()

    variable_reports_uns = (
        reports_to_uns_mapping(
            transfer_reports
        )
    )

    cdata.uns["bin_annotation_transfer"] = (
        sanitize_uns_for_h5ad(
            {
                "pipeline_version": (
                    PIPELINE_VERSION
                ),
                "output_schema_version": (
                    OUTPUT_SCHEMA_VERSION
                ),
                "source_zarr": str(p["zarr"]),
                "source_table": QC_TABLE_KEY,
                "bin_scope": (
                    TRANSFER_SPEC["bin_scope"]
                ),
                "assignment_report": (
                    assignment_report
                ),
                "variable_reports": (
                    variable_reports_uns
                ),
                # Human-readable exact copy of the external report.
                "variable_reports_json": json.dumps(
                    transfer_reports,
                    sort_keys=True,
                    default=str,
                ),
                "obs_transfer_validation": (
                    obs_transfer_validation
                ),
            }
        )
    )
    cdata.uns["cell_cycle_scoring"] = sanitize_none(cell_cycle_report)
    cdata.uns["resolvi_eligibility"] = {
        "minimum_counts": MIN_COUNTS_FOR_RESOLVI,
        "minimum_genes": MIN_GENES_FOR_RESOLVI,
        "apply_max_mito_filter": APPLY_MAX_MT_FILTER,
        "maximum_pct_mito": MAX_PCT_MT,
    }

    atomic_write_h5ad(cdata, p["annotated"])
    resolvi_input = cdata[cdata.obs["resolvi_eligible"].to_numpy()].copy()
    atomic_write_h5ad(resolvi_input, p["resolvi_input"])

    transfer_payload = {
        "sample": sample,
        "assignment_report": assignment_report,
        "variable_reports": transfer_reports,
        "obs_transfer_validation": (
            obs_transfer_validation
        ),
        "obs_transfer_schema_csv": str(
            p["obs_transfer_schema"]
        ),
        "cell_cycle_report": cell_cycle_report,
        "n_cells_all": int(cdata.n_obs),
        "n_cells_resolvi_eligible": int(resolvi_input.n_obs),
        "fraction_cells_resolvi_eligible": float(
            resolvi_input.n_obs / max(cdata.n_obs, 1)
        ),
    }
    atomic_json(transfer_payload, p["transfer_report"])

    qc_summary = pd.DataFrame(
        [
            {
                "sample": sample,
                "n_cells": int(cdata.n_obs),
                "n_resolvi_eligible": int(resolvi_input.n_obs),
                "median_total_counts": float(
                    cdata.obs["total_counts"].median()
                ),
                "median_genes": float(
                    cdata.obs["n_genes_by_counts"].median()
                ),
                "median_pct_mt": float(
                    cdata.obs["pct_counts_mt"].median()
                ),
                "fraction_high_mito_warning": float(
                    cdata.obs["qc_warning_high_mito"].mean()
                ),
                "fraction_annotation_bins_mapped": float(
                    assignment_report["fraction_bins_assigned"]
                ),
            }
        ]
    )
    qc_summary.to_csv(p["qc_summary"], index=False)

    (
        cdata.obs["phase"]
        .value_counts(dropna=False)
        .rename_axis("phase")
        .reset_index(name="n_cells")
        .to_csv(p["cell_cycle_summary"], index=False)
    )

    success = {
        "sample": sample,
        "completed": True,
        "annotated_h5ad": str(p["annotated"]),
        "resolvi_input_h5ad": str(p["resolvi_input"]),
        "n_cells": int(cdata.n_obs),
        "n_resolvi_eligible": int(resolvi_input.n_obs),
        "transfer_report": str(
            p["transfer_report"]
        ),
        "obs_transfer_schema_csv": str(
            p["obs_transfer_schema"]
        ),
        "pipeline_version": PIPELINE_VERSION,
        "output_schema_version": (
            OUTPUT_SCHEMA_VERSION
        ),
    }
    atomic_json(success, p["success"])
    p["failure"].unlink(missing_ok=True)

    del cdata, resolvi_input, gdf, source_obs, assignment
    gc.collect()
    return success


In [7]:
# ---------------------------------------------------------------------
# Run all samples and write a ResolVI source manifest
# ---------------------------------------------------------------------
results = {}
failures = {}

for _, row in manifest.iterrows():
    sample = str(row["sample"])
    print("\n" + "=" * 90)
    print("QC and annotation transfer:", sample)
    try:
        results[sample] = process_sample(row)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        failures[sample] = error
        p = paths_for_sample(row)
        atomic_json(
            {"sample": sample, "error": error},
            p["failure"],
        )
        traceback.print_exc(limit=25)
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        gc.collect()

summary = pd.DataFrame(results.values())
summary.to_csv(
    TRANSFER_ROOT / "all_samples_qc_transfer_summary.csv",
    index=False,
)
(
    TRANSFER_ROOT / "all_samples_qc_transfer_failures.json"
).write_text(json.dumps(failures, indent=2))

source_manifest = summary[
    ["sample", "annotated_h5ad", "resolvi_input_h5ad"]
].copy() if len(summary) else pd.DataFrame(
    columns=["sample", "annotated_h5ad", "resolvi_input_h5ad"]
)
source_manifest.to_csv(
    TRANSFER_ROOT / "resolvi_source_manifest.csv",
    index=False,
)

print("Completed:", sorted(results))
print("Failures:", json.dumps(failures, indent=2))
print("ResolVI manifest:", TRANSFER_ROOT / "resolvi_source_manifest.csv")

if failures:
    raise RuntimeError(
        "At least one sample failed QC/transfer. Completed samples remain usable."
    )



QC and annotation transfer: Ada-1
Reusing bin-to-cell assignment: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/tmp/tbio8110_stardist_proseg_resolvi_v1/03_cell_qc_obs_transfer/Ada-1/Ada-1_bin_to_proseg_cell_index.npy


/tmp/ipykernel_4477/2373918640.py:72: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(p["zarr"])


Transferred .obs schema:


/tmp/ipykernel_4477/3117292982.py:338: RuntimeWarning: All-NaN slice encountered
  maximum = np.nanmax(


,column,dtype,n_cells,n_missing,fraction_missing,n_unique_nonmissing,minimum,maximum
0,qc_class_dominant,category,430475,152,0.000353,5,NaN,NaN
1,qc_class_dominant_fraction,float32,430475,152,0.000353,984,0.25,1.0
2,qc_class_is_tie,bool,430475,0,0.000000,2,0.00,1.0
3,qc_class_n_valid_bins,int64,430475,0,0.000000,241,0.00,484.0
4,qc_class__fraction__tissue_high_transcript_high,float32,430475,152,0.000353,1242,0.00,1.0
5,qc_class__fraction__tissue_high_transcript_low,float32,430475,152,0.000353,1126,0.00,1.0
6,qc_class__fraction__tissue_low_transcript_high,float32,430475,152,0.000353,621,0.00,1.0
7,qc_class__fraction__tissue_low_transcript_low,float32,430475,152,0.000353,273,0.00,1.0


...

/tmp/ipykernel_4477/2373918640.py:72: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(p["zarr"])


Transferred .obs schema:


/tmp/ipykernel_4477/3117292982.py:338: RuntimeWarning: All-NaN slice encountered
  maximum = np.nanmax(


,column,dtype,n_cells,n_missing,fraction_missing,n_unique_nonmissing,minimum,maximum
0,qc_class_dominant,category,346563,28,0.000081,5,NaN,NaN
1,qc_class_dominant_fraction,float32,346563,28,0.000081,1366,0.25,1.0
2,qc_class_is_tie,bool,346563,0,0.000000,2,0.00,1.0
3,qc_class_n_valid_bins,int64,346563,0,0.000000,185,0.00,285.0
4,qc_class__fraction__tissue_high_transcript_high,float32,346563,28,0.000081,1910,0.00,1.0
5,qc_class__fraction__tissue_high_transcript_low,float32,346563,28,0.000081,1918,0.00,1.0
6,qc_class__fraction__tissue_low_transcript_high,float32,346563,28,0.000081,255,0.00,1.0
7,qc_class__fraction__tissue_low_transcript_low,float32,346563,28,0.000081,352,0.00,1.0


Completed: ['Ada-1', 'Ada-11R', 'Ada-12', 'Ada-14R', 'Ada-15', 'Ada-16', 'Ada-3R', 'Ada-4R', 'Ada-6', 'Ada-7', 'Ada-8', 'Ada-9']
Failures: {}
ResolVI manifest: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/03_cell_qc_obs_transfer/resolvi_source_manifest.csv
